In [1]:
import os
import re
import json
import logging
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional
from collections import Counter
from functools import partial
import random


import numpy as np
import pandas as pd


import matplotlib.pyplot as plt
import seaborn as sns


from tqdm.auto import tqdm


import joblib
import pickle


from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, average_precision_score, classification_report,
    precision_recall_curve, confusion_matrix
)


from scipy import sparse


try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold, MultilabelStratifiedShuffleSplit
except Exception:
    MultilabelStratifiedKFold = None
    MultilabelStratifiedShuffleSplit = None


try:

    from gensim.models import KeyedVectors
except Exception:
    KeyedVectors = None

try:
    import fasttext
    import fasttext.util
except Exception:
    fasttext = None


try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    from torch.optim import AdamW
except Exception:
    torch = None


try:
    from transformers import (
        AutoTokenizer, AutoModelForSequenceClassification,
        TrainingArguments, Trainer, DataCollatorWithPadding
    )
except Exception:
    AutoTokenizer = AutoModelForSequenceClassification = TrainingArguments = Trainer = DataCollatorWithPadding = None


import warnings
warnings.filterwarnings("ignore")


sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

C:\Users\valentin\.conda\envs\mirea-torch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def set_seed(seed: int = 42) -> None:
    # Фиксируем seed для воспроизводимости (насколько это возможно).
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

    # Настройки для более детерминированного поведения на GPU (если доступно).
    try:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass

    # Включаем детерминированные алгоритмы (PyTorch >=1.8). При необходимости окружение CUDA
    # может требовать настройки переменных CUBLAS_WORKSPACE_CONFIG или CPU реализаций.
    # try:
    #     torch.use_deterministic_algorithms(True)
    # except Exception:
    #     try:
    #         torch.set_deterministic(True)
    #     except Exception:
    #         pass

    # Сделаем генератор для DataLoader доступным глобально, чтобы все загрузчики были детерминированы
    global DATA_LOADER_GEN
    try:
        DATA_LOADER_GEN = torch.Generator()
        DATA_LOADER_GEN.manual_seed(seed)
    except Exception:
        DATA_LOADER_GEN = None

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [3]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("textdetox/multilingual_toxic_spans", split="ru")

In [4]:
df = pd.DataFrame(ds)

In [5]:
df.head()

,Sentence,Negative Connotations
0,во дебил ты пан-баран))))))))))))))))))). евре...,"дебил, пан-баран, еврейское тв, даун"
1,завтра в хабаровске вдвшники будут разгонять б...,быдло
2,"эта походу старое фото,в середине хряк телепуз...","хряк, телепузик, редкая скотина"
3,а что рядом других что мужиков рядом не оказал...,"ублюдку, табло, перевелись"
4,"хули вы бляди клянчите классы ,просто помните.","хули,бляди"


In [6]:
df.shape

(999, 2)

In [7]:
def tokenize(text):
    return re.findall(r'\w+|[^\w\s]', text.lower(), re.UNICODE)

def buildlabels(tokens, toxicwords):
    labels = [0] * len(tokens)

    toxiclist = [
        tokenize(word.strip())
        for word in toxicwords.split(',')
        if word.strip()
    ]

    for toxic in toxiclist:
        n = len(toxic)

        for i in range(len(tokens) - n + 1):
            if tokens[i:i+n] == toxic:
                for j in range(i, i+n):
                    labels[j] = 1

    return labels

df["tokens"] = df["Sentence"].apply(tokenize)

df["labels"] = df.apply(
    lambda row: buildlabels(
        row["tokens"],
        row["Negative Connotations"]
    ),
    axis=1
)

df[["tokens", "labels"]].head()

,tokens,labels
0,"[во, дебил, ты, пан, -, баран, ), ), ), ), ), ...","[0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,"[завтра, в, хабаровске, вдвшники, будут, разго...","[0, 0, 0, 0, 0, 0, 1]"
2,"[эта, походу, старое, фото, ,, в, середине, хр...","[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0]"
3,"[а, что, рядом, других, что, мужиков, рядом, н...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
4,"[хули, вы, бляди, клянчите, классы, ,, просто,...","[1, 0, 1, 0, 0, 0, 0, 0, 0]"


In [8]:
import pandas as pd

# Toxic lexicon dataset for trie
splits = {'ru': 'data/ru-00000-of-00001.parquet'}
df_lexicon = pd.read_parquet("hf://datasets/textdetox/multilingual_toxic_lexicon/" + splits["ru"])

In [9]:
df_lexicon.head()

,text
0,приебурясь
1,мудозвоном
2,пропиздюханную
3,перехуякавшихся
4,пидора́с


In [10]:
class TrieNode:
    __slots__ = ("children", "is_end")
    def __init__(self):
        self.children = {}
        self.is_end = False


class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word: str) -> None:
        node = self.root
        for ch in word:
            node = node.children.setdefault(ch, TrieNode())
        node.is_end = True

    def contains(self, word: str) -> bool:
        node = self.root
        for ch in word:
            node = node.children.get(ch)
            if node is None:
                return False
        return node.is_end

    def starts_with(self, prefix: str) -> bool:
        node = self.root
        for ch in prefix:
            node = node.children.get(ch)
            if node is None:
                return False
        return True

    def _iter_from(self, node: TrieNode, prefix: str):
        if node.is_end:
            yield prefix
        for ch, child in node.children.items():
            yield from self._iter_from(child, prefix + ch)

    def iter_words(self):
        yield from self._iter_from(self.root, "")

    def to_list(self):
        return list(self.iter_words())

    def __len__(self):
        return sum(1 for _ in self.iter_words())


import re

def build_trie_from_df(df, col: str = None, lower: bool = True, sep_regex: str = r"[\s,]+"):
    trie = Trie()
    if col is None:
        string_cols = [c for c in df.columns if df[c].dtype == object]
        cols = string_cols
    else:
        cols = [col]
    pattern = re.compile(sep_regex)
    for c in cols:
        for val in df[c].dropna().unique():
            if not isinstance(val, str):
                continue
            parts = [p for p in pattern.split(val) if p]
            for part in parts:
                w = part.strip()
                if lower:
                    w = w.lower()
                if w:
                    trie.insert(w)
    return trie

trie = build_trie_from_df(df_lexicon)
print("Words in trie:", len(trie))
# metrics for trie on validation
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

# prepare stratified split by presence of any toxic token in sentence
df = df.copy()
df['has_toxic'] = df['labels'].apply(lambda lbl: int(any(lbl)))
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['has_toxic'])

def predict_tokens_for_row(tokens, trie):
    return [1 if trie.contains(t.lower()) else 0 for t in tokens]

y_true = []
y_pred = []
y_score = []
for toks, labs in zip(val_df['tokens'], val_df['labels']):
    preds = predict_tokens_for_row(toks, trie)
    y_true.extend(labs)
    y_pred.extend(preds)
    y_score.extend(preds)

metrics = {}
metrics['accuracy'] = float(accuracy_score(y_true, y_pred))
metrics['precision'] = float(precision_score(y_true, y_pred, zero_division=0))
metrics['recall'] = float(recall_score(y_true, y_pred, zero_division=0))
try:
    metrics['roc_auc'] = float(roc_auc_score(y_true, y_score))
except Exception:
    metrics['roc_auc'] = None
metrics['model'] = 'Trie-lexicon'

metrics_df = pd.DataFrame([metrics])
print(metrics_df)



Words in trie: 140489
   accuracy  precision    recall   roc_auc         model
0  0.701734   0.234127  0.475806  0.607674  Trie-lexicon


In [11]:

from torch.utils.data import TensorDataset

class RNNTagger(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 100, hidden_size: int = 128,
                 num_layers: int = 1, dropout: float = 0.2, rnn_type: str = 'GRU'):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        rnn_type = rnn_type.upper()
        if rnn_type == 'LSTM':
            self.rnn = nn.LSTM(embed_dim, hidden_size, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers>1 else 0.0)
        elif rnn_type == 'RNN':
            self.rnn = nn.RNN(embed_dim, hidden_size, num_layers=num_layers, batch_first=True, nonlinearity='tanh', dropout=dropout if num_layers>1 else 0.0)
        else:
            self.rnn = nn.GRU(embed_dim, hidden_size, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers>1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.rnn(emb)
        out = self.dropout(out)
        logits = self.head(out).squeeze(-1)
        return logits

def build_vocab_from_tokens(token_lists, min_freq=1):
    freq = Counter()
    for toks in token_lists:
        for t in toks:
            freq[t] += 1
    word2idx = {'<PAD>': 0, '<UNK>': 1}
    idx = 2
    for w, c in freq.items():
        if c >= min_freq:
            word2idx[w] = idx
            idx += 1
    return word2idx

def tokens_to_padded_indices(tokens_list, w2i, max_len):
    seqs = []
    masks = []
    for toks in tokens_list:
        idxs = [w2i.get(t, w2i.get('<UNK>')) for t in toks][:max_len]
        mask = [1]*len(idxs)
        if len(idxs) < max_len:
            pad = [w2i['<PAD>']] * (max_len - len(idxs))
            idxs = idxs + pad
            mask = mask + [0]*(max_len - len(mask))
        seqs.append(idxs)
        masks.append(mask)
    return np.array(seqs, dtype=np.int64), np.array(masks, dtype=np.float32)

def labels_to_padded(labels_list, max_len):
    arr = []
    for labs in labels_list:
        labs = labs[:max_len]
        if len(labs) < max_len:
            labs = labs + [0]*(max_len - len(labs))
        arr.append(labs)
    return np.array(arr, dtype=np.float32)

# Prepare dataset (train/val split already exists earlier for trie). We'll reuse train_df and val_df if present.
if 'train_df' not in globals() or 'val_df' not in globals():
    df = df.copy()
    df['has_toxic'] = df['labels'].apply(lambda lbl: int(any(lbl)))
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['has_toxic'])

max_len = 100
word2idx = build_vocab_from_tokens(train_df['tokens'].tolist(), min_freq=1)

X_train_seq, X_train_mask = tokens_to_padded_indices(train_df['tokens'].tolist(), word2idx, max_len)
X_val_seq, X_val_mask = tokens_to_padded_indices(val_df['tokens'].tolist(), word2idx, max_len)
y_train = labels_to_padded(train_df['labels'].tolist(), max_len)
y_val = labels_to_padded(val_df['labels'].tolist(), max_len)

train_dataset = TensorDataset(torch.LongTensor(X_train_seq), torch.FloatTensor(y_train), torch.FloatTensor(X_train_mask))
val_dataset = TensorDataset(torch.LongTensor(X_val_seq), torch.FloatTensor(y_val), torch.FloatTensor(X_val_mask))

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

def train_and_evaluate(model, name, epochs=5, lr=1e-3):
    model = model.to(device)
    optimizer = AdamW(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss(reduction='none')
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_tokens = 0
        for Xb, yb, mb in train_loader:
            Xb = Xb.to(device)
            yb = yb.to(device)
            mb = mb.to(device)
            optimizer.zero_grad()
            logits = model(Xb)
            loss = loss_fn(logits, yb)
            loss = (loss * mb).sum() / (mb.sum() + 1e-9)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * mb.sum().item()
            total_tokens += mb.sum().item()
        avg_loss = total_loss / (total_tokens + 1e-9)
        # validation
        model.eval()
        y_true_all = []
        y_score_all = []
        with torch.no_grad():
            for Xb, yb, mb in val_loader:
                Xb = Xb.to(device)
                yb = yb.to(device)
                mb = mb.to(device)
                logits = model(Xb)
                probs = torch.sigmoid(logits)
                y_true_all.extend(yb[mb==1].cpu().numpy().tolist())
                y_score_all.extend(probs[mb==1].cpu().numpy().tolist())
        if len(set(y_true_all)) == 1:
            roc = None
        else:
            try:
                roc = float(roc_auc_score(y_true_all, y_score_all))
            except Exception:
                roc = None
        preds = [1 if s>=0.5 else 0 for s in y_score_all]
        acc = accuracy_score(y_true_all, preds) if len(y_true_all)>0 else 0.0
        prec = precision_score(y_true_all, preds, zero_division=0) if len(y_true_all)>0 else 0.0
        rec = recall_score(y_true_all, preds, zero_division=0) if len(y_true_all)>0 else 0.0
        print(f"{name} epoch {epoch+1}/{epochs} loss={avg_loss:.6f} val_acc={acc:.4f} prec={prec:.4f} rec={rec:.4f} roc_auc={roc}")
    # final metrics
    return {'model': name, 'accuracy': float(acc), 'precision': float(prec), 'recall': float(rec), 'roc_auc': roc}

metrics_list = []

# Train RNN, GRU, LSTM variants
for rnn_type in ['RNN', 'GRU', 'LSTM']:
    model = RNNTagger(vocab_size=len(word2idx), embed_dim=100, hidden_size=128, num_layers=2, dropout=0.3, rnn_type=rnn_type)
    metrics_res = train_and_evaluate(model, name=f"{rnn_type}-tagger", epochs=5, lr=1e-3)
    metrics_list.append(metrics_res)

metrics_df_trie = pd.DataFrame(metrics_list)
if 'metrics_df' in globals():
    metrics_df = pd.concat([metrics_df, metrics_df_trie], ignore_index=True)
else:
    metrics_df = metrics_df_trie

print(metrics_df)


RNN-tagger epoch 1/5 loss=0.455932 val_acc=0.8570 prec=0.5385 rec=0.0188 roc_auc=0.6770215102133608
RNN-tagger epoch 2/5 loss=0.366200 val_acc=0.8647 prec=0.7234 rec=0.0914 roc_auc=0.7271245386695302
RNN-tagger epoch 3/5 loss=0.337514 val_acc=0.8663 prec=0.6623 rec=0.1371 roc_auc=0.7589103894282162
RNN-tagger epoch 4/5 loss=0.315246 val_acc=0.8597 prec=0.5267 rec=0.2124 roc_auc=0.772746071133168
RNN-tagger epoch 5/5 loss=0.289207 val_acc=0.8682 prec=0.6136 rec=0.2177 roc_auc=0.7943923255892696
GRU-tagger epoch 1/5 loss=0.495429 val_acc=0.8543 prec=0.3125 rec=0.0134 roc_auc=0.6412983278433193
GRU-tagger epoch 2/5 loss=0.390936 val_acc=0.8555 prec=0.3846 rec=0.0134 roc_auc=0.7244853414208253
GRU-tagger epoch 3/5 loss=0.354029 val_acc=0.8574 prec=0.5385 rec=0.0376 roc_auc=0.762459550447666
GRU-tagger epoch 4/5 loss=0.323125 val_acc=0.8663 prec=0.7551 rec=0.0995 roc_auc=0.7952472682948066
GRU-tagger epoch 5/5 loss=0.291593 val_acc=0.8748 prec=0.7975 rec=0.1694 roc_auc=0.8118358171414198
LS